# Música y Salud Mental, Exploración de Datos

In [96]:
import pandas as pd

## Carga

In [97]:
df = pd.read_csv('data.csv')
print(f'Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
df.head(3)

Filas: 736 | Columnas: 33


,Timestamp,Age,Primary streaming service,Hours per day,While working,Instrumentalist,Composer,Fav genre,Exploratory,Foreign languages,...,Frequency [R&B],Frequency [Rap],Frequency [Rock],Frequency [Video game music],Anxiety,Depression,Insomnia,OCD,Music effects,Permissions
0,8/27/2022 19:29:02,18.0,Spotify,3.0,Yes,Yes,Yes,Latin,Yes,Yes,...,Sometimes,Very frequently,Never,Sometimes,3.0,0.0,1.0,0.0,NaN,I understand.
1,8/27/2022 19:57:31,63.0,Pandora,1.5,Yes,No,No,Rock,Yes,No,...,Sometimes,Rarely,Very frequently,Rarely,7.0,2.0,2.0,1.0,NaN,I understand.
2,8/27/2022 21:28:18,18.0,Spotify,4.0,No,No,No,Video game music,No,Yes,...,Never,Rarely,Rarely,Very frequently,7.0,7.0,10.0,2.0,No effect,I understand.


In [98]:
df.dtypes

Timestamp                           str
Age                             float64
Primary streaming service           str
Hours per day                   float64
While working                       str
Instrumentalist                     str
Composer                            str
Fav genre                           str
Exploratory                         str
Foreign languages                   str
BPM                             float64
Frequency [Classical]               str
Frequency [Country]                 str
Frequency [EDM]                     str
Frequency [Folk]                    str
Frequency [Gospel]                  str
Frequency [Hip hop]                 str
Frequency [Jazz]                    str
Frequency [K pop]                   str
Frequency [Latin]                   str
Frequency [Lofi]                    str
Frequency [Metal]                   str
Frequency [Pop]                     str
Frequency [R&B]                     str
Frequency [Rap]                     str


In [ ]:
mental_cols = ['Anxiety', 'Depression', 'Insomnia', 'OCD']
num_cols = mental_cols + ['Age', 'Hours per day', 'BPM']

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[mental_cols].describe().round(2)

,Anxiety,Depression,Insomnia,OCD
count,736.00,736.00,736.00,736.00
mean,5.84,4.80,3.74,2.64
std,2.79,3.03,3.09,2.84
min,0.00,0.00,0.00,0.00
25%,4.00,2.00,1.00,0.00
50%,6.00,5.00,3.00,2.00
75%,8.00,7.00,6.00,5.00
max,10.00,10.00,10.00,10.00


## Limpieza

In [100]:
print('Nulos por columna relevante:')
print(df[['Fav genre', 'Music effects'] + mental_cols].isnull().sum())

df_clean = df.dropna(subset=['Fav genre', 'Music effects'] + mental_cols).copy()
df_clean = df_clean[df_clean['Hours per day'] <= 20]
print(f'\nFilas tras limpieza: {df_clean.shape[0]}')

Nulos por columna relevante:
Fav genre        0
Music effects    8
Anxiety          0
Depression       0
Insomnia         0
OCD              0
dtype: int64

Filas tras limpieza: 725


## Generalidades

In [101]:
print("Distribución de efectos de la música:")
print(df_clean['Music effects'].value_counts())
print()
print("Géneros favoritos más frecuentes:")
print(df_clean['Fav genre'].value_counts())
print()
print("Promedios de salud mental por efecto musical:")
print(df_clean.groupby('Music effects')[mental_cols].mean().round(2))

Distribución de efectos de la música:
Music effects
Improve      540
No effect    168
Worsen        17
Name: count, dtype: int64

Géneros favoritos más frecuentes:
Fav genre
Rock                185
Pop                 114
Metal                88
Classical            53
Video game music     44
EDM                  36
R&B                  35
Hip hop              35
Folk                 29
Country              25
K pop                23
Jazz                 20
Rap                  20
Lofi                 10
Gospel                6
Latin                 2
Name: count, dtype: int64

Promedios de salud mental por efecto musical:
               Anxiety  Depression  Insomnia   OCD
Music effects                                     
Improve           6.05        4.87      3.75  2.72
No effect         5.15        4.40      3.69  2.38
Worsen            6.76        7.18      4.53  3.12


In [102]:
print("Promedio de depresión por género favorito (top 10):")
genre_dep = (df_clean.groupby('Fav genre')['Depression']
             .agg(['mean', 'median', 'count'])
             .sort_values('median', ascending=False)
             .round(2))
print(genre_dep)

Promedio de depresión por género favorito (top 10):
                  mean  median  count
Fav genre                            
Lofi              6.60     8.0     10
Hip hop           5.80     7.0     35
Metal             5.07     6.0     88
Rock              5.29     6.0    185
EDM               5.11     5.5     36
Folk              5.14     5.0     29
Country           4.32     5.0     25
Classical         4.08     5.0     53
Pop               4.49     5.0    114
Latin             4.50     4.5      2
Video game music  4.48     4.5     44
Jazz              4.50     4.0     20
R&B               3.83     4.0     35
K pop             4.30     4.0     23
Rap               4.15     3.0     20
Gospel            2.67     1.0      6


In [103]:
print("Horas promedio de escucha por efecto musical:")
print(df_clean.groupby('Music effects')['Hours per day'].mean().round(2))
print()
print("Correlación entre horas de escucha y salud mental:")
print(df_clean[['Hours per day'] + mental_cols].corr()['Hours per day'].round(3))

Horas promedio de escucha por efecto musical:
Music effects
Improve      3.59
No effect    3.32
Worsen       2.76
Name: Hours per day, dtype: float64

Correlación entre horas de escucha y salud mental:
Hours per day    1.000
Anxiety          0.073
Depression       0.151
Insomnia         0.163
OCD              0.146
Name: Hours per day, dtype: float64
